# 03 — Statistical Analysis (All Stations)

This notebook extracts key statistical characteristics (mean, variability,
and trend) from the prepared monthly water quality time series.
The extracted statistics will be used as the foundation for statistical
mock prediction in subsequent steps.


Cell 2 — Import Libraries

In [1]:
import os
import json
import pandas as pd
import numpy as np


Cell 3 — Define Paths

In [2]:
PREP_DIR = "../data/prepared"
STATS_DIR = "../data/stats"

os.makedirs(STATS_DIR, exist_ok=True)


Cell 4 — Define Parameters to Analyze

In [3]:
PARAMETERS = [
    "secchi",
    "chlorophyll_a",
    "tsi",
    "turbidity",
    "salinity",
    "do",
    "ph"
]


Cell 5 — Helper Function: Trend Calculation ⭐

In [4]:
def compute_trend(series):
    """
    Compute linear trend (slope per time step).
    Returns 0 if insufficient data.
    """
    y = series.dropna().values
    if len(y) < 2:
        return 0.0

    x = np.arange(len(y))
    slope = np.polyfit(x, y, 1)[0]
    return float(slope)


Cell 6 — Loop วิเคราะห์ทุกสถานี

In [5]:
for file in os.listdir(PREP_DIR):
    if not file.endswith("_prepared.csv"):
        continue

    station = file.replace("_prepared.csv", "")
    path = os.path.join(PREP_DIR, file)

    df = pd.read_csv(path, parse_dates=["date"], index_col="date")

    stats = {}

    for param in PARAMETERS:
        if param not in df.columns:
            continue

        s = df[param].dropna()

        stats[param] = {
            "mean": float(s.mean()),
            "std": float(s.std()),
            "min": float(s.min()),
            "max": float(s.max()),
            "trend": compute_trend(s)
        }

    # save stats
    out_path = os.path.join(STATS_DIR, f"{station}_stats.json")
    with open(out_path, "w") as f:
        json.dump(stats, f, indent=2)

    print(f"📊 Stats extracted: {station}")


📊 Stats extracted: CP01
📊 Stats extracted: LS01
📊 Stats extracted: LS03
📊 Stats extracted: PN01
📊 Stats extracted: SK01
📊 Stats extracted: SK06
📊 Stats extracted: TP011
📊 Stats extracted: TP01
📊 Stats extracted: TP04


Cell 7 — Inspect One Station (Sanity Check)

In [6]:
with open("../data/stats/CP01_stats.json") as f:
    cp01_stats = json.load(f)

cp01_stats


{'secchi': {'mean': 1.5763340345392007,
  'std': 0.21977617401027932,
  'min': 1.2012175406473455,
  'max': 2.2272445576395072,
  'trend': -0.004310403514335821},
 'chlorophyll_a': {'mean': 14.108886326320098,
  'std': 2.528407001707987,
  'min': 8.63370130700132,
  'max': 24.87197053539824,
  'trend': -0.013173176263868158},
 'tsi': {'mean': 56.109750602192406,
  'std': 1.755023280089857,
  'min': 51.40034580699572,
  'max': 61.7002867918137,
  'trend': -0.0016366697033900457},
 'turbidity': {'mean': 114.38120270226983,
  'std': 12.35556900261873,
  'min': 99.9064561719501,
  'max': 145.3386810029175,
  'trend': -0.13808537319068903},
 'salinity': {'mean': 0.06040833842124535,
  'std': 0.03901897605705807,
  'min': 0.0118704568826966,
  'max': 0.1852096943705563,
  'trend': -0.0008187265683893001},
 'do': {'mean': 9.57726026235523,
  'std': 0.00040530926918366756,
  'min': 9.576332226594186,
  'max': 9.578446014152023,
  'trend': -1.0819560888461075e-06},
 'ph': {'mean': 7.00755843538

Cell 8 — Quick Interpretation (Optional แต่ดีมาก)

In [7]:
for p, v in cp01_stats.items():
    print(f"{p:15s} | mean={v['mean']:.2f}, std={v['std']:.2f}, trend={v['trend']:.3f}")


secchi          | mean=1.58, std=0.22, trend=-0.004
chlorophyll_a   | mean=14.11, std=2.53, trend=-0.013
tsi             | mean=56.11, std=1.76, trend=-0.002
turbidity       | mean=114.38, std=12.36, trend=-0.138
salinity        | mean=0.06, std=0.04, trend=-0.001
do              | mean=9.58, std=0.00, trend=-0.000
ph              | mean=7.01, std=0.55, trend=0.010
